# Algorithms

In the [Introduction notebook](https://saminulh.github.io/dpstats/library/tutorials/tasks/), we discussed how a task defines the types of its input, metadata, and output. Each algorithm belongs to one task and implements a `run` method using that task’s types. For mean estimation, the signature is:

```python
def run(
    self,
    metadata: MeanEstimationMetadata,
    input: MeanEstimationInput,
    budget: DPBudget,
    seed: int,
) -> MeanEstimationOutput:
    ...
```

We implement algorithms as classes rather than functions to manage hyperparameters and give each configuration a stable identity. We configure an algorithm by passing its hyperparameters to the constructor.

## Built-in algorithms

Every task comes with a set of built-in algorithms that are implemented in the library. The `list_built_in_algorithms` function returns a dictionary with the names of the algorithms as keys and their corresponding classes as values. Below we see the built-in algorithms for the [MeanEstimation](https://saminulh.github.io/dpstats/mean_estimation/) task.

In [ ]:
from dpstats.tasks import MeanEstimation, list_built_in_algorithms

list(list_built_in_algorithms(MeanEstimation).keys())


### Constructing an algorithm

Let's choose `CoinPress` as our algorithm. To see the hyperparameters for a given algorithm, you can call its `parameter_info` function. You may also visit that algorithm’s [docs page](https://saminulh.github.io/dpstats/mean_estimation/algorithms/CoinPress/) for reference.

In [ ]:
from dpstats.tasks.mean_estimation.algorithms import CoinPress

CoinPress.parameter_info()


Algorithms are classes and we can instantiate them by passing in their hyperparameters (or using their defaults by not passing anything in). Algorithms of the same class with identical hyperparameters have the same UUID, which is used later for sweeping the benchmark.

In [ ]:
default_coinpress = CoinPress()
coinpress_four_iters = CoinPress(num_iters=4)
coinpress_two_iters = CoinPress(num_iters=2)

print("default:", default_coinpress)
print("four_iters:", coinpress_four_iters)
print("two_iters:", coinpress_two_iters)
print(
    "different configurations, different UUIDs:",
    default_coinpress.uuid != coinpress_four_iters.uuid,
)
print(
    "same configuration, same UUID:", default_coinpress.uuid == coinpress_two_iters.uuid
)


### Running a built-in algorithm

Once the algorithm is instantiated, running it is simply a matter of passing the required arguments into the algorithm's `run` function.

In [ ]:
import numpy as np

from dpstats.budgets import ApproxDPBudget
from dpstats.domains import BoxDomain
from dpstats.tasks.mean_estimation import (
    MeanEstimationInput,
    MeanEstimationMetadata,
)

metadata = MeanEstimationMetadata(
    n_samples=300, domain=BoxDomain(bounds=np.array([[0.0, 1.0]]))
)
valid_input = MeanEstimationInput(X=np.tile([[0.2], [0.5], [0.8]], (100, 1)))
metadata.validate_input(valid_input)
budget = ApproxDPBudget(eps=1.0, delta=1e-6)

built_in_output = default_coinpress.run(
    metadata=metadata, input=valid_input, budget=budget, seed=42
)
print("CoinPress mean_est:", built_in_output.mean_est)


## Implementing your own algorithm

Implementing an algorithm in our library requires three things:

1. It must extend `BaseAlgorithm`.
2. It must declare its task by setting the `TASK` attribute.
3. It must implement the `run` method with the declared task’s types.

You can define hyperparameters as class attributes with [parameter_field](https://saminulh.github.io/dpstats/library/reference/parameters/dpstats.parameters.parameter_field/).

Here we walk through a simple example implementing the Gaussian mechanism for mean estimation.

### Gaussian mechanism

As a brief primer on the Gaussian mechanism, this method computes the sample mean and adds Gaussian noise with sufficient scale as to satisfy the privacy guarantee. This scale grows proportionally with the $\ell_2$-sensitivity, which is how much adding/removing one sample can change the sample mean in $\ell_2$-norm. This sensitivity is only finite if the domain of possible values for the sample is itself bounded, otherwise adding increasingly large points can change the sample mean arbitrarily.

Below we implement a minimal version of [GaussianMechanism](https://saminulh.github.io/dpstats/mean_estimation/algorithms/GaussianMechanism/) for mean estimation. We use the `domain` supplied in the metadata to compute the sensitivity. As the Gaussian mechanism is common, the `accounting` library supplies a function for computing the necessary noise-scale for a given budget and sensitivity. We go into handling budgets in more depth in the [Budgets notebook](https://saminulh.github.io/dpstats/library/tutorials/privacy_budgets_and_accounting/).

In [ ]:
import numpy as np

from dpstats.algorithm import BaseAlgorithm
from dpstats.budgets import DPBudget, accounting
from dpstats.exceptions import InvalidRegime
from dpstats.tasks import MeanEstimation
from dpstats.tasks.mean_estimation import (
    MeanEstimationInput,
    MeanEstimationMetadata,
    MeanEstimationOutput,
)


class OurGaussianMechanism(BaseAlgorithm):
    TASK = MeanEstimation

    def run(
        self,
        metadata: MeanEstimationMetadata,
        input: MeanEstimationInput,
        budget: DPBudget,
        seed: int,
    ) -> MeanEstimationOutput:

        _, radius = metadata.domain.l2_enclosing_ball()
        if not np.isfinite(radius):
            raise InvalidRegime("A finite enclosing radius is required.")
        # With public n and enclosing-ball center c, the centered query is
        # c + sum(X - c) / n. Adding/removing one record changes it by at most
        # radius / n; on this validated input, it equals the sample mean.
        sensitivity = radius / metadata.n_samples

        sigma = accounting.calibrate_gaussian_scale(budget, sensitivity)

        mean = np.mean(input.X, axis=0)
        rng = np.random.default_rng(seed)
        return MeanEstimationOutput(
            mean_est=mean + rng.normal(0, sigma, size=mean.shape)
        )


We can now run this algorithm on our previous toy input.

In [ ]:
algorithm = OurGaussianMechanism()
output = algorithm.run(metadata=metadata, input=valid_input, budget=budget, seed=42)
print("OurGaussianMechanism mean_est:", output.mean_est)


As discussed before, the Gaussian mechanism requires the domain to be bounded. If the domain is unbounded, the algorithm raises an [InvalidRegime](https://saminulh.github.io/dpstats/library/reference/exceptions/dpstats.exceptions.InvalidRegime/) exception, meaning that it cannot solve the supplied problem under these conditions.

In [ ]:
from dpstats.domains import EuclideanSpace

unbounded_metadata = MeanEstimationMetadata(n_samples=300, domain=EuclideanSpace(dim=1))
unbounded_metadata.validate_input(valid_input)
print("The input still satisfies the metadata.")

try:
    algorithm.run(
        metadata=unbounded_metadata, input=valid_input, budget=budget, seed=42
    )
except InvalidRegime as error:
    print("InvalidRegime:", error)


## Composite Algorithms

We saw that the bare Gaussian mechanism algorithm immediately fails on instances with an unbounded domain. For such cases, one may wish to spend some of the privacy budget estimating the bounds on the data, clip the input to those bounds, and then run the Gaussian mechanism on this processed input that does have bounded domain.

In our library, we represent algorithms with pre-processing stages as [`CompositeAlgorithm`](https://saminulh.github.io/dpstats/library/reference/algorithm/dpstats.algorithm.CompositeAlgorithm/) objects and the individual processing steps as [`Transformations`](https://saminulh.github.io/dpstats/library/reference/algorithm/dpstats.algorithm.Transformation/). Here, the [`ApproxBounds`](https://saminulh.github.io/dpstats/mean_estimation/transformations/ApproxBounds/) transformation privately estimates bounds on dimensions missing bounds and clips the input to those bounds.

The composite algorithm of ApproxBounds + OurGaussianMechanism now runs on the unbounded input.

In [ ]:
from dpstats.algorithm import CompositeAlgorithm
from dpstats.tasks.mean_estimation.transformations import ApproxBounds

composite_algorithm = CompositeAlgorithm(
    transformations=(ApproxBounds(),), algorithm=OurGaussianMechanism()
)
composite_output = composite_algorithm.run(
    metadata=unbounded_metadata, input=valid_input, budget=budget, seed=42
)
print("ApproxBounds + OurGaussianMechanism mean_est:", composite_output.mean_est)


We dive into these in more detail in the [Transformations notebook](https://saminulh.github.io/dpstats/library/tutorials/extend_with_transformations/).

## Next steps

Next: [Domains](https://saminulh.github.io/dpstats/library/tutorials/domains/).